# GATraj-aWTA on Google Colab
Select **Runtime > Change runtime type > GPU** and run the cells in order.

This notebook uses the same GATraj architecture and 20-mode Laplace decoder, with an optional annealed winner-takes-all regression loss. It is an adaptation of Xu et al., ICRA 2025, not a new loss invented here. No accuracy improvement is guaranteed or claimed before evaluation.

It follows the RiskMotion notebook workflow: mount Drive, configure a fold, train in chunks, automatically resume from Drive, and evaluate the fixed final checkpoint. The default runs **both baseline and aWTA**; this approximately doubles training time versus one model. Both use identical common settings. There is no RiskMotion module or attention correction in this repository.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone and install
Code runs on local Colab storage. Completed checkpoints are written to Drive every epoch with temporary-file replacement. A CPU regression check runs before GPU training, including checkpoint save/load and continuation.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

DRIVE_ROOT = Path('/content/drive/MyDrive')
assert DRIVE_ROOT.is_dir(), 'Mount Drive first.'
REPO = Path('/content/GATraj-aWTA')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Kedar2002/GATraj-aWTA.git', str(REPO)], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO, check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU runtime.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-c',
    "import torch, unittest; torch.set_num_threads(2); "
    "suite=unittest.defaultTestLoader.discover('tests'); "
    "result=unittest.TextTestRunner(verbosity=2).run(suite); "
    "raise SystemExit(0 if result.wasSuccessful() else 1)"], check=True)

## Configure one fold
Fold IDs: **0 ETH, 1 Hotel, 2 Zara1, 3 Zara2, 4 University**.

`VARIANTS = ['baseline', 'awta']` trains the matched pair in separate directories. Use `['awta']` for the modified model only, or `['baseline']` for baseline only. Keep all common settings and seeds fixed for a comparison.

`VALIDATION_SET = -1` uses the standard four-scene training / one-scene test protocol. For an optional validation experiment, choose another scene index. Then both selected scenes are excluded from training: three scenes train and the designated validation scene is evaluated. The true test scene is not evaluated during that experiment. These pilot scores must not be reported as standard test results.

After validation, retrain the selected variant from scratch with `VALIDATION_SET = -1`. The validation comparison retains baseline if aWTA does not improve both metrics without a tradeoff. This is a validation rule, not a guarantee on unseen test data.

Defaults for temperature are starting hypotheses, not tuned benchmark settings. Annealing runs from 0.5 m to 0.01 m over the first 30% of absolute epochs, followed by original hard WTA. The full cosine learning-rate schedule remains 1,000 epochs. Setting `STOP_AT_EPOCH = 1` checks startup without changing either schedule.

In [ ]:
TEST_SET = 0
SEED = 1
VARIANTS = ['baseline', 'awta']
VALIDATION_SET = -1  # Optional pilot: another scene index, different from TEST_SET.
AWTA_INITIAL = 0.5
AWTA_FINAL = 0.01
AWTA_FRACTION = 0.3

TOTAL_EPOCHS = 1000
STOP_AT_EPOCH = TOTAL_EPOCHS
EPOCHS_PER_RUN = 100
PRUNE_INTERMEDIATE_CHECKPOINTS = True
KEEP_EVERY = 100
OUTPUT_DIR = DRIVE_ROOT / 'GATraj-aWTA' / 'runs'

## Train or automatically resume from Drive
Rerun with identical settings after a restart. Each variant has its own directory and metadata compatibility checks. Existing baseline/RiskMotion weights are not automatically reused. The latest numbered file must load successfully; unreadable files cause an explicit error rather than a hidden restart.

Every epoch is saved to Drive. After a successful chunk, optional cleanup keeps every 100th checkpoint and the latest one. Turn pruning off to keep all epochs. An unfinished epoch or interrupted Drive write cannot be preserved. Avoid running two sessions against the same experiment directory.

In [ ]:
# Ask the current repository to resolve each experiment name, avoiding stale imports.
assert 0 < STOP_AT_EPOCH <= TOTAL_EPOCHS
assert EPOCHS_PER_RUN > 0 and KEEP_EVERY > 0
assert set(VARIANTS) <= {'baseline', 'awta'} and len(VARIANTS) == len(set(VARIANTS)) and VARIANTS
assert VALIDATION_SET == -1 or (0 <= VALIDATION_SET <= 4 and VALIDATION_SET != TEST_SET)

common_args = [
    '--test_set', str(TEST_SET), '--seed', str(SEED),
    '--validation_set', str(VALIDATION_SET), '--num_epochs', str(TOTAL_EPOCHS),
    '--x_encoder_layers', '3', '--eta_min', '1e-5', '--batch_size', '32',
    '--learning_rate', '5e-4', '--randomRotate', 'True', '--final_mode', '20',
    '--neighbor_thred', '10', '--using_cuda', 'True', '--gpu', '0', '--clip', '1',
    '--pass_time', '2', '--ifGaussian', 'False', '--SR', 'True', '--input_offset', 'True',
    '--awta_initial', str(AWTA_INITIAL), '--awta_final', str(AWTA_FINAL),
    '--awta_fraction', str(AWTA_FRACTION), '--save_base_dir', str(OUTPUT_DIR),
]

def stream(command):
    with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    if result:
        raise RuntimeError(f'Process exited with code {result}; see the actual error above.')

def checkpoint_files(run_dir, run_name):
    checkpoints = {}
    for path in run_dir.glob(f'{run_name}_*.tar'):
        suffix = path.stem.rsplit('_', 1)[-1]
        if suffix.isdigit() and int(suffix) > 0:
            checkpoints[int(suffix)] = path
    return checkpoints

def validate_checkpoint(path, epoch):
    # Full loading is appropriate for your own trusted training checkpoints.
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    assert int(ckpt['epoch']) == epoch, f'Epoch mismatch: {path}'
    required = ('state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'rng_state', 'experiment_config')
    missing = [key for key in required if key not in ckpt]
    if missing:
        raise RuntimeError(f'{path} lacks {missing}; this isolated experiment requires its own new checkpoints.')
    del ckpt

runs = {}
for variant in VARIANTS:
    variant_args = [*common_args, '--awta', str(variant == 'awta')]
    resolver = 'from train import get_parser, experiment_name; print(experiment_name(get_parser().parse_args()))'
    name = subprocess.check_output([sys.executable, '-c', resolver, *variant_args], cwd=REPO, text=True).strip()
    run_dir = OUTPUT_DIR / str(TEST_SET) / name
    runs[variant] = (name, run_dir, variant_args)
    available = checkpoint_files(run_dir, name)
    completed = max(available, default=0)
    if completed > TOTAL_EPOCHS:
        raise RuntimeError('Checkpoint exceeds configured horizon; check the experiment directory.')
    if completed:
        validate_checkpoint(available[completed], completed)
    print(f'Variant: {variant}; resume after epoch {completed}; checkpoints: {run_dir}')
    while completed < STOP_AT_EPOCH:
        count = min(EPOCHS_PER_RUN, STOP_AT_EPOCH - completed)
        stream([sys.executable, '-u', 'train.py', *variant_args,
                '--epochs_per_run', str(count), '--load_model', str(completed)])
        advanced = max(checkpoint_files(run_dir, name), default=0)
        if advanced != completed + count:
            raise RuntimeError('Expected checkpoint was not saved; inspect training output.')
        validate_checkpoint(run_dir / f'{name}_{advanced}.tar', advanced)
        completed = advanced
        if PRUNE_INTERMEDIATE_CHECKPOINTS:
            for epoch, path in checkpoint_files(run_dir, name).items():
                if epoch < completed and epoch % KEEP_EVERY:
                    path.unlink()
        print(f'{variant}: saved to Drive through epoch {completed}')

## Evaluate the predetermined final checkpoint
Each model produces exactly 20 trajectories. The report includes minADE20/minFDE20, density subsets, parameters, latency and a top-1 collision diagnostic. Compare timing on the same GPU. The model architecture and parameter count are identical in the two arms.

The next cell evaluates either the designated pilot validation scene or the true test scene, never both in one run. It requires the fixed final epoch; startup runs do not select a best test checkpoint.

In [ ]:
reports = {}
for variant, (name, run_dir, variant_args) in runs.items():
    path = run_dir / f'{name}_{TOTAL_EPOCHS}.tar'
    if not path.exists():
        raise RuntimeError('Finish TOTAL_EPOCHS before the final evaluation.')
    validate_checkpoint(path, TOTAL_EPOCHS)
    stream([sys.executable, '-u', 'evaluate.py', *variant_args,
            '--phase', 'test', '--load_model', str(TOTAL_EPOCHS)])
    reports[variant] = run_dir / f'evaluation_{TOTAL_EPOCHS}.json'
    report = json.loads(reports[variant].read_text())
    print(variant, report['split'], report['groups']['all'])

if {'baseline', 'awta'} <= reports.keys():
    stage = f'validation{VALIDATION_SET}' if VALIDATION_SET >= 0 else 'test'
    comparison_path = OUTPUT_DIR / str(TEST_SET) / f'comparison_{stage}_s{SEED}_{runs["awta"][0]}.json'
    stream([sys.executable, 'compare.py', str(reports['baseline']), str(reports['awta']),
            '--output', str(comparison_path)])
    print('Comparison saved:', comparison_path)

## Report the experiment
Repeat final experiments for all five folds with paired seeds (for example 1, 2 and 3). Average the five folds equally for each seed, then report seed variability. Preserve baseline results even if aWTA is worse. Validation-based fallback reduces selection risk but cannot guarantee better test ADE/FDE.

Persistent files: `MyDrive/GATraj-aWTA/runs/<fold>/<experiment>/`. Each new experiment trains from scratch and remains separate from your base GATraj and RiskMotion checkpoints.